# How to Use EPIC Instrumental Background Files -- Part 1: Images
<hr style="border: 2px solid #f5bf03" />

- **Description:** The tutorial shows how to produce a Filter Wheel Closed image, and illustrates how it can be used to correct the science data for instrumental background.
- **Level:** Intermediate
- **Data:** XMM observation of SN 1006-1, an extended source (obsid=0555630101)
- **Requirements:** Must be run using pySAS version 2.2.8 or higher.
- **Credit:** Ryan Tanner (December 2025), based on an <a href="https://www.cosmos.esa.int/web/xmm-newton/sas-thread-background">ESA SOC SAS Tutorial</a>
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 30 June 2026, for SAS v22.1 and pySAS v2.5.0

<hr style="border: 2px solid #f5bf03" />

## 1. Introduction

EPIC instrumental background files are produced with the filter wheel equipped in the CLOSED position. These exposures can be used to model and treat the EPIC camera background component, which are composed by:

- Electronic readout noise (at lowest energies)
- High energy particles producing charge directly in the CCDs
- Particle induced X-rays (continuum and fluorescent lines), generated inside the camera

These components are know collectivelly as the Quiescent Particle Background (QPB). A Filter Wheel Closed (FWC) repository is available through the XMM-Newton Science Operations Centre from the EPIC Background Analysis web pages. Since SAS v16, it is possible to access this FWC repository through the SAS task `evqpb` to produce a tailored FWC event file suitable for a given science observation. `evqpb` only deals with EPIC-pn Full Frame and Extended Full Frame mode exposures and EPIC-MOS Full Frame mode exposures. It is recommended not to use these files for other modes since the instrumental noise depends on the exposure mode used.

This tutorial explains how to create one of this FWC event files for a particular science observation, and how to create background corrected images. In Part 2 we do the same but for spectra. For this purpose, the XMM-Newton observation Obs ID 0555630101 will be used, which corresponds to the extended source SN 1006-1.

The general work flow is as follows:

1. Generate calibrated science event lists.
2. Generate good time interval (GTI) files that can be used to calculate the '`Livetime`' for the observation.
3. Generate filter wheel closed (FWC) event lists.
4. Apply GTI files to FWC event lists.
5. Filter the science event lists using standard methods.
6. Filter the FWC event lists using the *exact* same methods used on the science event lists.
7. Subtract the FWC event lists from the science event lists to generate background corrected science event lists.

#### SAS Tasks to be Used

- `evqpb`[(Documentation for evqpb)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/evqpb/index.html "evqpb Documentation")
- `evselect`[(Documentation for evselect)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/evselect/index.html "evselect Documentation")
- `espfilt`[(Documentation for espfilt)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/espfilt/index.html "espfilt Documentation")
- `tabgtigen`[(Documentation for tabgtigen)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/tabgtigen/index.html "tabgtigen Documentation")

#### Useful Links

- [`pysas` Documentation](https://github.com/XMMGOF/pysas_docs "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads/ "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/cgi-bin/Feedback "Helpdesk") - Link to form to contact the XMM-Newton GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

## 2. Setup

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask

# Useful imports
import re, glob, shutil

# HEASoftpy import
import heasoftpy as hsp

# Astropy imports
from astropy.io import fits
from astropy.visualization import astropy_mpl_style
from astropy.wcs import WCS

# Imports for plotting
import matplotlib.pyplot as plt
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
obsid  = '0555630101'
my_obs = pysas.ObsID(obsid)

In [ ]:
my_obs.basic_setup(overwrite=False,
                   run_rgsproc=False,
                   epproc_args=['-V 1'],
                   emproc_args=['-V 1'])

***
The cell below contains a number of functions that will be used throughout this notebook.

In [ ]:
def filter_event_list(in_event_list,
                      out_event_list = 'filtered_event_list.fits',
                      pi_min  = 500,
                      pi_max  = 12000,
                      pattern = None):

    with fits.open(in_event_list) as hdu:
        instrument = hdu[0].header['INSTRUME']

    if instrument == 'EPN':
        filter = 'XMMEA_EP'
        if pattern is None: pattern = 4
    elif 'EMOS' in instrument:
        filter = 'XMMEA_EM'
        if pattern is None: pattern = 12

    # Filter expression
    expression = '(PATTERN in [0:{pattern}])&&(PI in [{pi_min}:{pi_max}])&&(FLAG == 0)&&#{filter}'.format(filter=filter,pattern=pattern,pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'           : in_event_list, 
              'withfilteredset' : 'yes', 
              'expression'      : expression, 
              'filteredset'     : out_event_list, 
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes'}
    
    MyTask('evselect', inargs).run()

def make_hires_image(in_event_list,
                     zcolumn  = None,
                     out_image='image.fits',
                     output   = True):

    inargs = {'table'         : in_event_list, 
              'withimageset'  : 'yes',
              'imageset'      : out_image,
              'imagebinning'  : 'binSize',
              'xcolumn'       : 'X',
              'ycolumn'       : 'Y',
              'ximagebinsize' : 80,
              'yimagebinsize' : 80}

    if zcolumn is not None:
        inargs['zcolumn'] = zcolumn
    
    MyTask('evselect', inargs, output_to_terminal = output).run()

def plot_images(left_image,center_image,right_image,titles, vmin=0.1, vmax=30.0):
    """
    Takes file names for three FITS images and a list containing
    the titles for the three plots.

    Plots the images in a row. All images must have the same WCS.
    """
    hdu = []
    hdu.append(fits.open(left_image)[0])
    hdu.append(fits.open(center_image)[0])
    hdu.append(fits.open(right_image)[0])
    
    fig, axes = plt.subplots(1, 3,subplot_kw={'projection': WCS(hdu[0].header)}, figsize=(15, 15))
    plt.grid(color='blue', ls='solid')
    
    for i, ax in enumerate(axes.flat):
        ax.set_facecolor("black")
        ax.grid(color='blue', ls='solid')
        ax.imshow(hdu[i].data, origin='lower', norm='log', vmin=vmin, vmax=vmax)
        ax.title.set_text(titles[i])
        ax.coords[0].set_axislabel('RA')
        if i != 0:
            ax.coords[1].set_ticklabel_visible(False)
            ax.coords[1].set_axislabel('')
        else:
            ax.coords[1].set_axislabel('Dec')
    
    plt.subplots_adjust(wspace=0)

***
The cell below will make a series of dictionaries containing filenames that will be used throughout this notebook.

In [ ]:
event_lists = {}

for filename in my_obs.files['PNevt_list']:
    if re.search('.*EPN_S.*ImagingEvts.ds$',filename):
        event_lists['PN'] = filename
for filename in my_obs.files['M1evt_list']:
    if re.search('.*EMOS1_S.*ImagingEvts.ds$',filename):
        event_lists['M1'] = filename
for filename in my_obs.files['M2evt_list']:
    if re.search('.*EMOS2_S.*ImagingEvts.ds$',filename):
        event_lists['M2'] = filename

time_filtered_evts = {}
clean_event_lists  = {}
hi_res_images      = {}

gti_files          = {}

fwc_files        = {}
fwc_files_time   = {}
fwc_files_clean  = {}
fwc_hi_res_images= {}

bkgrd_corr_image   = {}

for instrument in event_lists.keys():
    clean_event_lists[instrument] = f'{instrument}_event_list_clean.fits'
    hi_res_images[instrument]     = f'{instrument}_image.fits'
    fwc_files[instrument]         = f'{instrument}_fwc.fits'
    fwc_files_time[instrument]    = f'{instrument}_fwc_time_filtered.fits'
    fwc_files_clean[instrument]   = f'{instrument}_fwc_clean.fits'
    fwc_hi_res_images[instrument] = f'{instrument}_fwc_image.fits'
    bkgrd_corr_image[instrument]  = f'{instrument}_bkgrd_corrected_image.fits'
    
pn_ltcv_file  = 'pn_light_curve.fits'
pn_filt_rate  = 'pnS003-allevc.fits'
pn_gti_file   = 'pnS003-gti.fits'
attitude_file = glob.glob('*AttHk.ds')[0]

## 3. Generate Good Time Interval Files

### 3.1 Check the Unfiltered Event Lists

Let's start by plotting the unfiltered event lists to see what we are working with.

In [ ]:
for inst,file in event_lists.items():
    my_obs.quick_eplot(file, vmax=100)

### 3.2 Filter for Flaring Using 'espfilt'

We will use the SAS task `espfilt` to generate good time interval (GTI) files and to filter for solar flares.

<div class="alert alert-block alert-info">
    <b>Note:</b> This Obs ID doesn't have significant contamination from solar flares, so a GTI file isn't strictly necessary, but we show it here for completeness.
</div>

<div class="alert alert-block alert-warning">
    <b>Warning:</b> <tt>espfilt</tt> will work just fine on this Obs ID for both <tt>MOS</tt> event lists, but will fail for the <tt>pn</tt> event list. We will have to generate a <tt>GTI</tt> and filtered event list for the <tt>pn</tt> using a more direct approach. We will cover that in the next section.
</div>

In [ ]:
for inst,file in event_lists.items():
    if inst == 'PN':
        rangescale = 15
    elif (inst == 'M1') or (inst == 'M2'):
        rangescale = 6
    inargs = {'eventfile'  : file,
              'rangescale' : rangescale,
              'elow'       : 500,
              'ehigh'      : 11995}
    MyTask('espfilt', inargs).run()

### 3.3 Generating a GTI File for the pn

As mentioned in the previous section, `espfilt` will fail for the pn for this particular Obs ID. So we will have to generate the GTI file directly. Let's start by looking at the light curve for the pn.

In [ ]:
my_obs.quick_lcplot(event_lists['PN'], light_curve_file=pn_ltcv_file)

We don't see much contamination, but for this tutorial we will still filter it and generate a GTI file. Let's filter using `RATE <= 88`.

In [ ]:
inargs = {'table'      : pn_ltcv_file, 
          'gtiset'     : pn_gti_file,
          'timecolumn' : 'TIME', 
          'expression' : "'(RATE <= 88)'"}

MyTask('tabgtigen', inargs).run()

inargs = {'table'           : event_lists['PN'],
          'withfilteredset' : 'yes', 
          'expression'      : "'GTI({0},TIME)'".format(pn_gti_file), 
          'filteredset'     : pn_filt_rate,
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes'}

MyTask('evselect', inargs).run()

Now we can check the final light curve for the pn.

In [ ]:
my_obs.quick_lcplot(pn_filt_rate)

Now we can collect all the filenames so that we can use them later.

In [ ]:
gti_event_lists = glob.glob('*allevc.fits')
attitude_file = glob.glob('*AttHk.ds')[0]
gti_file_list = glob.glob('*-gti.fits')

for event_list in gti_event_lists:
    if re.search('.*pn.*',event_list):
        time_filtered_evts['PN'] = event_list
    if re.search('.*mos1.*',event_list):
        time_filtered_evts['M1'] = event_list
    if re.search('.*mos2.*',event_list):
        time_filtered_evts['M2'] = event_list

for filename in gti_file_list:
    if re.search('.*pn.*',filename):
        gti_files['PN'] = filename
    if re.search('.*mos1.*',filename):
        gti_files['M1'] = filename
    if re.search('.*mos2.*',filename):
        gti_files['M2'] = filename

## 4. Generate Filter Wheel Closed Event Lists

### 4.1 Use 'evqpb' to Make FWC Event Lists

To generate the output FWC event file, events from the repository are selected based on proximity in time to the science observation and around the central time of the exposure. The parameter `exposurefactor` indicates the time to be accumulated until an exposure of `exposurefactor*Livetime` is achieved. In this case the generated FWC event file will have an exposure time that is a factor 2.0 higher than the science observation. The instrumental background varies over time, hence it is recommended to use a reasonable `exposurefactor` so that one can assume that the instrumental background remains constant over the period of time covered by the science exposure.

The SAS task `evqpb` needs to make use of the attitude history file corresponding to the science observation, parameter `attfile`, to recast the FWC events in space to match those of the science observation. The parameter `outset` defines the names of the corresponding FWC event files.

In [ ]:
for inst, event_list in time_filtered_evts.items():
    inargs = {'table'          : event_list, 
              'exposurefactor' : 2.0,
              'attfile'        : attitude_file, 
              'outset'         : fwc_files[inst],
              'options'        : '-V 1'}
    
    MyTask('evqpb', inargs).run()

### 4.2 Apply the GTI Files to the FWC Event Lists

If you previously generated GTI files for your observation then you will need to apply those files to the FWC Event Lists. We can do this using `evselect` just like we did for the `pn` above.

In [ ]:
for inst, event_list in fwc_files.items():
    inargs = {'table'           : event_list,
              'withfilteredset' : 'yes', 
              'expression'      : "'GTI({0},TIME)'".format(gti_files[inst]), 
              'filteredset'     : fwc_files_time[inst],
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes'}
    
    MyTask('evselect', inargs).run()

## 5. Generate Background Corrected Images

### 5.1 Filter All Event Lists

From here on, any filtering you do on the science data event lists you must do on the FWC event lists. The filtering can be anything we want to filter on, we could filter using a different pattern selection or different energy range. Again, the only important aspect to keep in mind is that the filtering should be the same on the science data and FWC event files.

In [ ]:
for inst in time_filtered_evts.keys():
    # Filter the science data event lists
    filter_event_list(time_filtered_evts[inst], out_event_list=clean_event_lists[inst])
    # Filter the FWC event lists
    filter_event_list(fwc_files_time[inst], out_event_list=fwc_files_clean[inst])

We can check the filtered science data event lists.

In [ ]:
for inst,file in clean_event_lists.items():
    my_obs.quick_eplot(file, vmax=50)

### 5.2 Generate FITS Image Files

The use of the parameter `zcolumn` is important at this stage when generating the image of the FWC event file. This parameter indicates that a weight needs to be applied to every single event when generating the image. The value of the parameter has to be the name of an exiting column in the event file. In this case, we use the column named `EWEIGHT`. This column contains for every single event the ratio of the `Sience_Exposure / FWC_Exposure` and varies on a CCD basis. By using the `zcolumn` parameter, we ensure that the produced FWC image is correctly scaled in time to the science image and that is ready to be subtracted from the science image.

In [ ]:
for inst in clean_event_lists.keys():
    make_hires_image(clean_event_lists[inst], out_image = hi_res_images[inst])
    make_hires_image(fwc_files_clean[inst], out_image = fwc_hi_res_images[inst], zcolumn = 'EWEIGHT')

### 5.3 Subtract FWC Events from Science Events

Now we can subtract the FWC event lists from the science event lists to create background corrected science event lists.

We use the HEASoft routine `farith` to do the subtraction. We then plot the uncorrected science events, the FWC events, and the background subtracted science events for all three EPIC cameras.

In [ ]:
# This won't produce any output because it is inside of a 'for' loop
for inst in bkgrd_corr_image.keys():
    hsp.farith(infil1   = hi_res_images[inst],
               infil2   = fwc_hi_res_images[inst],
               outfil   = bkgrd_corr_image[inst],
               ops      = 'SUB',
               noprompt = True,
               clobber  = True)

In [ ]:
for inst in hi_res_images.keys():
    titles = [f'{inst} Science Data',f'{inst} FWC Data',f'{inst} Corrected Data']
    plot_images(hi_res_images[inst],fwc_hi_res_images[inst],bkgrd_corr_image[inst],titles)